In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)
from xgboost import XGBClassifier

from src.data.generate import generate
from src.data.preprocess import preprocess
from src.config.config import (
    FEATURES,
    TARGET,
    RF_PARAM_DISTRIBUTIONS,
    XGB_PARAM_DISTRIBUTIONS,
    RF_PARAM_GRID,
    XGB_PARAM_GRID,
)

RANDOM_STATE = 42
TEST_SIZE = 0.2
VALIDATION_SIZE = 0.2
RF_SEARCH_ITERS = 50
XGB_SEARCH_ITERS = 50

raw_df = generate(n=10000, seed=RANDOM_STATE, save=False)
df = preprocess(raw_df, save=False)

X = df[FEATURES]
y = df[TARGET]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_train_val,
)

results = []
trained_models = []

print(f'Train size: {len(X_train)} | Validation size: {len(X_val)} | Test size: {len(X_test)}')
print(f'No-show rate in test: {y_test.mean():.2%}')

Train size: 6400 | Validation size: 1600 | Test size: 2000
No-show rate in test: 23.70%


In [3]:
def tune_threshold_on_validation(model, X_val, y_val, thresholds=None):
    if thresholds is None:
        thresholds = np.linspace(0.1, 0.9, 50)

    y_val_proba = model.predict_proba(X_val)[:, 1]

    best_threshold = 0.5
    best_f1 = -1.0

    for threshold in thresholds:
        y_val_pred = (y_val_proba >= threshold).astype(int)
        score = f1_score(y_val, y_val_pred)
        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    return best_threshold, best_f1


def evaluate_on_test(name, model, X_test, y_test, threshold):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    print(f'\n{"-"*50}')
    print(f'EXPERIMENT: {name}')
    print(f'Threshold (from validation): {threshold:.3f}')
    print(f'{"-"*50}')
    print(f'Accuracy: {acc:.4f}')
    print(f'ROC-AUC: {auc:.4f}')
    print(f'F1: {f1:.4f}')
    print('Confusion Matrix:')
    print(f'    TN={cm[0][0]}  FP={cm[0][1]}')
    print(f'    FN={cm[1][0]}  TP={cm[1][1]}')
    print(classification_report(y_test, y_pred, target_names=['Show', 'No-Show']))

    return {
        'name': name,
        'accuracy': acc,
        'f1': f1,
        'roc_auc': auc,
        'threshold': threshold,
    }


def plot_feature_importances(models, feature_names, top_n=15, figsize=(12, 5)):
    importance_frames = []

    for item in models:
        model = item['model']
        if not hasattr(model, 'feature_importances_'):
            continue

        label = item['name']
        if item.get('params'):
            params_text = ', '.join(f"{k}={v}" for k, v in item['params'].items())
            label = f'{label} ({params_text})'

        frame = pd.DataFrame(
            {
                'feature': feature_names,
                'importance': model.feature_importances_,
                'algorithm': label,
            }
        )
        importance_frames.append(frame)

    if not importance_frames:
        print('No models with feature importances were provided.')
        return

    all_importances = pd.concat(importance_frames, ignore_index=True)

    for algorithm in all_importances['algorithm'].unique():
        subset = (
            all_importances[all_importances['algorithm'] == algorithm]
            .sort_values('importance', ascending=False)
            .head(top_n)
        )

        plt.figure(figsize=figsize)
        sns.barplot(
            data=subset,
            x='importance',
            y='feature',
            hue='feature',
            palette='viridis',
            legend=False,
        )
        plt.title(f'Feature Importances - {algorithm}')
        plt.xlabel('Importance')
        plt.ylabel('Feature')
        plt.tight_layout()
        plt.show()


def plot_correlation_heatmap(dataframe, columns=None, figsize=(10, 8), cmap='coolwarm'):
    corr_source = dataframe[columns] if columns is not None else dataframe
    corr = corr_source.corr(numeric_only=True)

    plt.figure(figsize=figsize)
    sns.heatmap(corr, annot=True, fmt='.2f', cmap=cmap, square=True, linewidths=0.5)
    plt.title('Feature Correlation Heatmap')
    plt.tight_layout()
    plt.show()


def plot_class_distribution(target_series, labels=('Show', 'No-Show')):
    class_counts = target_series.value_counts().sort_index()
    total = class_counts.sum()

    if len(labels) == len(class_counts):
        x_labels = labels
    else:
        x_labels = [str(idx) for idx in class_counts.index]

    percentages = class_counts / total * 100
    dist_df = pd.DataFrame({'class': x_labels, 'count': class_counts.values})

    plt.figure(figsize=(8, 5))
    ax = sns.barplot(
        data=dist_df,
        x='class',
        y='count',
        hue='class',
        palette='Set2',
        legend=False,
    )
    plt.title('Class Distribution')
    plt.xlabel('Class')
    plt.ylabel('Count')

    for i, (count, pct) in enumerate(zip(class_counts.values, percentages.values)):
        ax.text(i, count, f'{count} ({pct:.1f}%)', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()


print('Helpers ready!')

Helpers ready!


# RandomizedSearch
Tried randomized search cv but came to the conclusion that this is used on large datasets that are bigger and bigger datasets are when it exceeds the capacity of traditional data processing tools to handle efficiently.

In [4]:
cv_rf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=RF_PARAM_DISTRIBUTIONS,
    n_iter=RF_SEARCH_ITERS,
    scoring='f1',
    cv=cv_rf,
    verbose=1,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_
rf_threshold, rf_val_f1 = tune_threshold_on_validation(best_rf, X_val, y_val)
rf_result = evaluate_on_test('RandomForest RandomizedSearchCV', best_rf, X_test, y_test, rf_threshold)
rf_result['val_f1_threshold_tuning'] = rf_val_f1
rf_result['cv_best_f1'] = rf_search.best_score_
rf_result['best_params'] = rf_search.best_params_
results.append(rf_result)
trained_models.append({'name': 'random_forest_best', 'model': best_rf, 'params': rf_search.best_params_})

cv_xgb = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
xgb_search = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=XGB_PARAM_DISTRIBUTIONS,
    n_iter=XGB_SEARCH_ITERS,
    scoring='f1',
    cv=cv_xgb,
    verbose=1,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
xgb_search.fit(X_train, y_train)

best_xgb_params = xgb_search.best_params_
best_xgb = XGBClassifier(
    **best_xgb_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=50,
)
best_xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

xgb_threshold, xgb_val_f1 = tune_threshold_on_validation(best_xgb, X_val, y_val)
xgb_result = evaluate_on_test('XGBoost RandomizedSearchCV + early stopping', best_xgb, X_test, y_test, xgb_threshold)
xgb_result['val_f1_threshold_tuning'] = xgb_val_f1
xgb_result['cv_best_f1'] = xgb_search.best_score_
xgb_result['best_params'] = best_xgb_params
results.append(xgb_result)
trained_models.append({'name': 'xgboost_best', 'model': best_xgb, 'params': best_xgb_params})

Fitting 5 folds for each of 50 candidates, totalling 250 fits

--------------------------------------------------
EXPERIMENT: RandomForest RandomizedSearchCV
Threshold (from validation): 0.508
--------------------------------------------------
Accuracy: 0.6085
ROC-AUC: 0.6229
F1: 0.4000
Confusion Matrix:
    TN=956  FP=570
    FN=213  TP=261
              precision    recall  f1-score   support

        Show       0.82      0.63      0.71      1526
     No-Show       0.31      0.55      0.40       474

    accuracy                           0.61      2000
   macro avg       0.57      0.59      0.55      2000
weighted avg       0.70      0.61      0.64      2000

Fitting 3 folds for each of 50 candidates, totalling 150 fits

--------------------------------------------------
EXPERIMENT: XGBoost RandomizedSearchCV + early stopping
Threshold (from validation): 0.492
--------------------------------------------------
Accuracy: 0.4780
ROC-AUC: 0.6316
F1: 0.4181
Confusion Matrix:
    TN=581 

# GridSearchCV refinement


In [5]:
cv_rf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rf_grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=RF_PARAM_GRID,
    scoring='f1',
    cv=cv_rf,
    verbose=1,
    n_jobs=-1,
)
rf_grid_search.fit(X_train, y_train)

best_rf_grid = rf_grid_search.best_estimator_
rf_grid_threshold, rf_grid_val_f1 = tune_threshold_on_validation(best_rf_grid, X_val, y_val)
rf_grid_result = evaluate_on_test('RandomForest GridSearchCV', best_rf_grid, X_test, y_test, rf_grid_threshold)
rf_grid_result['val_f1_threshold_tuning'] = rf_grid_val_f1
rf_grid_result['cv_best_f1'] = rf_grid_search.best_score_
rf_grid_result['best_params'] = rf_grid_search.best_params_
results.append(rf_grid_result)
trained_models.append({'name': 'random_forest_grid_best', 'model': best_rf_grid, 'params': rf_grid_search.best_params_})

cv_xgb = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
xgb_grid_search = GridSearchCV(
    estimator=XGBClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid=XGB_PARAM_GRID,
    scoring='f1',
    cv=cv_xgb,
    verbose=1,
    n_jobs=-1,
)
xgb_grid_search.fit(X_train, y_train)

best_xgb_grid_params = xgb_grid_search.best_params_
best_xgb_grid = XGBClassifier(
    **best_xgb_grid_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=50,
)
best_xgb_grid.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

xgb_grid_threshold, xgb_grid_val_f1 = tune_threshold_on_validation(best_xgb_grid, X_val, y_val)
xgb_grid_result = evaluate_on_test('XGBoost GridSearchCV + early stopping', best_xgb_grid, X_test, y_test, xgb_grid_threshold)
xgb_grid_result['val_f1_threshold_tuning'] = xgb_grid_val_f1
xgb_grid_result['cv_best_f1'] = xgb_grid_search.best_score_
xgb_grid_result['best_params'] = best_xgb_grid_params
results.append(xgb_grid_result)
trained_models.append({'name': 'xgboost_grid_best', 'model': best_xgb_grid, 'params': best_xgb_grid_params})

Fitting 5 folds for each of 288 candidates, totalling 1440 fits

--------------------------------------------------
EXPERIMENT: RandomForest GridSearchCV
Threshold (from validation): 0.427
--------------------------------------------------
Accuracy: 0.4825
ROC-AUC: 0.6228
F1: 0.4089
Confusion Matrix:
    TN=607  FP=919
    FN=116  TP=358
              precision    recall  f1-score   support

        Show       0.84      0.40      0.54      1526
     No-Show       0.28      0.76      0.41       474

    accuracy                           0.48      2000
   macro avg       0.56      0.58      0.47      2000
weighted avg       0.71      0.48      0.51      2000

Fitting 3 folds for each of 59049 candidates, totalling 177147 fits


KeyboardInterrupt: 

In [4]:
summary = pd.DataFrame(results)
summary = summary.sort_values(['f1', 'roc_auc'], ascending=False).reset_index(drop=True)

experiment_path = Path('artifacts/experiments/search_comparison_summary.csv')
experiment_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(experiment_path, index=False)

print(f'Saved experiment summary to: {experiment_path}')
summary

Saved experiment summary to: artifacts\experiments\randomized_search_summary.csv


,name,accuracy,f1,roc_auc,threshold,val_f1_threshold_tuning,cv_best_f1,best_params
0,XGBoost RandomizedSearchCV + early stopping,0.4780,0.41806,0.631557,0.491837,0.421274,0.419676,"{'colsample_bytree': 0.8630451569201374, 'eval..."
1,RandomForest RandomizedSearchCV,0.6085,0.40000,0.622873,0.508163,0.425000,0.398650,"{'bootstrap': False, 'class_weight': 'balanced..."


In [ ]:
plot_feature_importances(trained_models, FEATURES, top_n=len(FEATURES), figsize=(11, 4))
plot_correlation_heatmap(df[FEATURES + [TARGET]], figsize=(12, 9))
plot_class_distribution(df[TARGET], labels=('Show', 'No-Show'))